# # Chicago Taxi Trips — week of 11–17 September 2023
#
# **Summary** — _to be filled in once the analysis is complete._

# ## 1. Data loading and cleaning
#
# All figures in this notebook describe **reported trips** in `trips.csv` (Mon 2023-09-11 to Sun 2023-09-17), not total taxi market volume — the City notes that not every trip is reported.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# One restrained house style: a single blue for magnitude, recessive grid, no box.
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#9a9992", "#2b2b2a"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left", "legend.frameon": False,
})
FIG = Path("figures/analysis")
FIG.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv("trips.csv", parse_dates=["trip_start_timestamp", "trip_end_timestamp"])
print("shape:", raw.shape)
print("trip_id unique:", raw.trip_id.is_unique, "| fully duplicated rows:", raw.duplicated().sum())
print("start:", raw.trip_start_timestamp.min(), "->", raw.trip_start_timestamp.max())

shape: (135768, 23)
trip_id unique: True | fully duplicated rows: 0
start: 2023-09-11 00:00:00 -> 2023-09-17 23:45:00


In [2]:
miss = raw.isna().sum()
print("Columns with missing values (n, % of 135,768 rows):")
print(pd.DataFrame({"n_missing": miss, "pct": miss / len(raw) * 100})[miss > 0])

Columns with missing values (n, % of 135,768 rows):
                            n_missing   pct
trip_end_timestamp                  1  0.00
trip_seconds                       33  0.02
pickup_census_tract             71404 52.59
dropoff_census_tract            73842 54.39
pickup_community_area            3815  2.81
dropoff_community_area          13794 10.16
fare                              324  0.24
tips                              324  0.24
tolls                             324  0.24
extras                            324  0.24
trip_total                        324  0.24
pickup_centroid_latitude         3775  2.78
pickup_centroid_longitude        3775  2.78
pickup_centroid_location         3775  2.78
dropoff_centroid_latitude       12872  9.48
dropoff_centroid_longitude      12872  9.48
dropoff_centroid_location       12872  9.48


In [3]:
r = raw
mph = r.trip_miles / (r.trip_seconds / 3600)
checks = {
    "trip_seconds == 0": (r.trip_seconds == 0).sum(),
    "trip_seconds missing": r.trip_seconds.isna().sum(),
    "trip_seconds > 6 hours": (r.trip_seconds > 6 * 3600).sum(),
    "trip_miles == 0": (r.trip_miles == 0).sum(),
    "trip_miles == 0 but fare > $10": ((r.trip_miles == 0) & (r.fare > 10)).sum(),
    "trip_miles > 100": (r.trip_miles > 100).sum(),
    "implied speed > 80 mph (miles & seconds > 0)": (mph > 80).sum(),
    "fare/tips/tolls/extras/trip_total all missing": r.trip_total.isna().sum(),
    "trip_total == 0": (r.trip_total == 0).sum(),
    "fare > $200": (r.fare > 200).sum(),
    "trip_total > $500": (r.trip_total > 500).sum(),
    "tolls > $100": (r.tolls > 100).sum(),
    "extras > $100": (r.extras > 100).sum(),
}
print(pd.Series(checks, name="rows").to_frame().assign(pct=lambda d: d.rows / len(r) * 100).to_string(float_format=lambda x: f"{x:,.2f}"))
print()
print("Most extreme money rows (by trip_total):")
print(r.sort_values("trip_total", ascending=False)[["trip_seconds","trip_miles","fare","tips","tolls","extras","trip_total","payment_type"]].head(6))

                                                rows   pct
trip_seconds == 0                               2731  2.01
trip_seconds missing                              33  0.02
trip_seconds > 6 hours                            98  0.07
trip_miles == 0                                13993 10.31
trip_miles == 0 but fare > $10                  5778  4.26
trip_miles > 100                                  11  0.01
implied speed > 80 mph (miles & seconds > 0)     228  0.17
fare/tips/tolls/extras/trip_total all missing    324  0.24
trip_total == 0                                  134  0.10
fare > $200                                       49  0.04
trip_total > $500                                  8  0.01
tolls > $100                                       3  0.00
extras > $100                                     15  0.01

Most extreme money rows (by trip_total):
        trip_seconds  trip_miles   fare  tips    tolls   extras  trip_total payment_type
124235      1,080.00        0.10  11.25  0.

In [4]:
parts = r[["fare", "tips", "tolls", "extras"]].sum(axis=1, min_count=4)
gap = (r.trip_total - parts).round(2)
print("trip_total minus (fare + tips + tolls + extras), rows with all money fields present:", gap.notna().sum())
print(gap.value_counts().head(8).rename("rows").to_frame().T.to_string())
print()
print("Gap < 0:", (gap < -0.005).sum(), "| gap > $3:", (gap > 3.005).sum(), "| gap > $5:", (gap > 5).sum())
print("Rows with a non-zero gap, by payment type (share of that type):")
print((gap.abs() > 0.005).groupby(r.payment_type).mean().round(3).to_dict())
print(f"Total gap ${gap.sum():,.0f} = {gap.sum() / r.trip_total.sum():.2%} of reported trip_total")

trip_total minus (fare + tips + tolls + extras), rows with all money fields present: 135444
       0.00   0.50  3.00  2.50  1.00  1.95  1.50  2.00
rows  87247  46522   804   751   109     5     4     2

Gap < 0: 0 | gap > $3: 0 | gap > $5: 0
Rows with a non-zero gap, by payment type (share of that type):
{'Cash': 0.017, 'Credit Card': 0.719, 'Dispute': 0.0, 'Mobile': 0.224, 'No Charge': 0.0, 'Prcard': 0.001, 'Unknown': 0.0}
Total gap $27,679 = 0.68% of reported trip_total


In [5]:
vc = r.company.value_counts()
print("Distinct company labels:", len(vc))
print("Labels that look like spelling variants of one another, with trips and distinct taxis:")
variants = ["Taxicab Insurance Agency Llc", "Taxicab Insurance Agency, LLC", "Choice Taxi Association", "Choice Taxi Association Inc",
            "Taxi Affiliation Services", "Taxi Affiliation Services Llc - Yell"]
tx = r.groupby("company").taxi_id.agg(trips="size", taxis="nunique").loc[variants]
print(tx)
sets = r.groupby("company").taxi_id.agg(set)
for a, b in [variants[0:2], variants[2:4], variants[4:6]]:
    print(f"taxi_ids shared by '{a}' and '{b}':", len(sets[a] & sets[b]))
print("taxis reporting under more than one company label:", (r.groupby("taxi_id").company.nunique() > 1).sum(), "of", r.taxi_id.nunique())

Distinct company labels: 32
Labels that look like spelling variants of one another, with trips and distinct taxis:
                                      trips  taxis
company                                           
Taxicab Insurance Agency Llc          13914    294
Taxicab Insurance Agency, LLC          2379     62
Choice Taxi Association                2022     50
Choice Taxi Association Inc            1064     20
Taxi Affiliation Services             25727    457
Taxi Affiliation Services Llc - Yell     75      2
taxi_ids shared by 'Taxicab Insurance Agency Llc' and 'Taxicab Insurance Agency, LLC': 0
taxi_ids shared by 'Choice Taxi Association' and 'Choice Taxi Association Inc': 0
taxi_ids shared by 'Taxi Affiliation Services' and 'Taxi Affiliation Services Llc - Yell': 0


taxis reporting under more than one company label: 3 of 2656
